In [ ]:
import geopandas as gpd
import pandas as pd

import cafo_iowa.db.session as s
import cafo_iowa.db.models as m

import os

import datetime

# Analyzing Barn Annotations Outside of Facility Parcels

This notebook identifies barn annotations from the CF model that fall outside of known facility parcel boundaries. This could indicate:

1. Missing or incorrect facility parcel data
2. False positive barn detections by the model
3. Real barns that are not part of a registered CAFO facility

We'll analyze these cases by:
1. Finding all barn annotations that don't intersect with facility parcels
2. Creating annotated images of these locations for manual review
3. Recording the lat/lon coordinates for field verification

We send this data to ReGrid to obtain permits



In [ ]:
# Load annotations and facilities from database
annotations = gpd.read_postgis(
    "SELECT * FROM processed.cf_annotations WHERE type = 'segment' AND label != 'Blank' and geometry is not null",
    s.get_engine(),
    geom_col="geometry",
)

facilities = gpd.read_postgis(
    "SELECT * FROM processed.facilities", s.get_engine(), geom_col="geometry"
)

# Spatial join barns with parcels
# Each barn that doesn't match any parcel will have NaN values
annotations_w_facilities = gpd.sjoin(
    annotations, facilities, how="left", predicate="intersects"
)

# Count barns with no parcel match (NaN values)
annotations_wo_facilities = annotations_w_facilities[
    annotations_w_facilities["index_right"].isna()
]
num_annoations_outside = len(annotations_wo_facilities)

print(f"Number of annotations outside of any facility: {num_annoations_outside}")

# Get unique NAIP tile IDs for these barns
naip_tile_ids = annotations_wo_facilities["naip_qt_id"].unique()
print(
    f"\nNumber of NAIP tile IDs containing annotations outside of facilities: {len(naip_tile_ids)}"
)

In [ ]:
# Get today's date
date_today = datetime.datetime.now().strftime("%Y-%m-%d")

# Convert barns outside parcels to WGS84 (EPSG:4326) for lat/lon
annotations_wo_facilities = annotations_wo_facilities.to_crs(epsg=4326)

# Get corners of each barn polygon
annotations_wo_facilities["lat_min"] = annotations_wo_facilities.geometry.bounds.miny
annotations_wo_facilities["lat_max"] = annotations_wo_facilities.geometry.bounds.maxy
annotations_wo_facilities["lon_min"] = annotations_wo_facilities.geometry.bounds.minx
annotations_wo_facilities["lon_max"] = annotations_wo_facilities.geometry.bounds.maxx

# Create rows for each corner combination
corners = []
for _, row in annotations_wo_facilities.iterrows():
    corners.extend(
        [
            {"id": row["id"], "lat": row["lat_min"], "lon": row["lon_min"]},
            {"id": row["id"], "lat": row["lat_max"], "lon": row["lon_max"]},
        ]
    )

# Convert to DataFrame and save
corners_df = pd.DataFrame(corners)
corners_df.to_csv(f"data/temp/{date_today}_annotations_with_no_parcel.csv", index=False)